# Phase 3: Abstract Decisions

Score selected species–PMID pairs with the configured decision provider. Jev probabilities are retained in the decision ledger; this notebook derives an `accepted` flag using the configurable `confidence_threshold` (default 0.75).

Run the configuration cell in notebook 00 first to change Phase 3 settings. Set `RUN_PHASE_OVERRIDE = True` in the setup cell (or set `run_phase: true` in `configs/notebook_parameters.yaml`) to make API calls.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import ensure_parent, resolve_path
from pathogen_atlas_v2.llm.pathogen_evidence import apply_confidence_threshold, phase3_preflight, run_phase3

NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase3_abstract_decisions", NOTEBOOK_CONFIG_PATH)
RUN_PHASE_OVERRIDE = None  # Set True to run now, False to skip, or None to use YAML.
RUN_PHASE = bool(PARAMS.get("run_phase", False)) if RUN_PHASE_OVERRIDE is None else bool(RUN_PHASE_OVERRIDE)
RUN_ID = str(PARAMS.get("run_id", "jev-human-pathogen-evidence-v1"))
DECISION_PATH = resolve_path(PARAMS.get("decision_path", "data/03_decisions/parsed/pathogen_evidence_jev.parquet"))
ACCEPTED_DECISIONS_PATH = resolve_path(PARAMS.get("accepted_decisions_path", "data/03_decisions/parsed/pathogen_evidence_jev_accepted.parquet"))
CONFIDENCE_THRESHOLD = float(PARAMS.get("confidence_threshold", 0.75))

## Preflight

This reads Phase 2 release tables and the local decision ledger. It does not call Jev. Check the pending pair count, selected model, question version, and maximum call count before enabling the run.

In [ ]:
PREFLIGHT = phase3_preflight(PARAMS)
PREFLIGHT["confidence_threshold"] = CONFIDENCE_THRESHOLD
display(pd.DataFrame([PREFLIGHT]))

## Run Jev decisions

The runner skips successful rows whose run ID, input hash, model, and question version match. Failed responses remain eligible for a later retry. Keep `max_calls` at 25 for a pilot; set it to `None` only when you intend to process all pending pairs.

In [ ]:
if RUN_PHASE:
    if PREFLIGHT.get("status") != "ready":
        raise RuntimeError(f"Phase 3 is not ready: {PREFLIGHT}")
    PHASE3_RESULT = run_phase3(PARAMS, log=True)
else:
    PHASE3_RESULT = {
        "status": "not_run",
        "reason": "Set run_phase: true in the Phase 3 notebook parameters to call Jev.",
        "preflight": PREFLIGHT,
    }
display(PHASE3_RESULT)

## Apply the confidence threshold

The notebook labels a completed decision as accepted when `p_yes >= confidence_threshold`. The default is 0.75 and can be changed in `configs/notebook_parameters.yaml`. Failed or missing decisions remain unaccepted. The raw probability ledger is unchanged; accepted rows are written to a separate Parquet file.


In [ ]:
if DECISION_PATH.exists():
    DECISIONS = pd.read_parquet(DECISION_PATH)
    if "run_id" in DECISIONS.columns:
        CURRENT_RUN = DECISIONS.loc[DECISIONS["run_id"].astype(str).eq(RUN_ID)].copy()
    else:
        CURRENT_RUN = DECISIONS.iloc[0:0].copy()
    CURRENT_RUN = apply_confidence_threshold(CURRENT_RUN, CONFIDENCE_THRESHOLD)
    ACCEPTED_DECISIONS = CURRENT_RUN.loc[CURRENT_RUN["accepted"].fillna(False)].copy()
    ensure_parent(ACCEPTED_DECISIONS_PATH)
    ACCEPTED_DECISIONS.to_parquet(ACCEPTED_DECISIONS_PATH, index=False)
    if CURRENT_RUN.empty:
        print(f"No saved decisions for run {RUN_ID!r} yet.")
    else:
        display(CURRENT_RUN["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))
        display(CURRENT_RUN["accepted"].value_counts(dropna=False).rename_axis("accepted at threshold").to_frame("rows"))
        display(ACCEPTED_DECISIONS.head(20))
        print(f"Accepted {len(ACCEPTED_DECISIONS)} of {len(CURRENT_RUN)} decisions at p_yes >= {CONFIDENCE_THRESHOLD:.2f}.")
    print(f"Accepted decisions saved to: {ACCEPTED_DECISIONS_PATH}")
else:
    print(f"Decision ledger does not exist yet: {DECISION_PATH}")
